# Copyright 2022 Cognite AS

## Import the Libraries and Modules

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

utils = str(Path("../utils").resolve())
if utils not in sys.path:
    sys.path.append(utils)

from cognite_auth import interactive_client

## Create the Cognite Client

In [2]:
client = interactive_client()

C:\Users\JoeO'Bryant\OneDrive - Cognite AS\projects\cognite_python_sdk_training\using-cognite-python-sdk\.venv\Lib\site-packages\ipykernel\ipkernel.py:772: UserWarning: You are using version='7.86.0' of the SDK, however version='7.87.0' is available. To suppress this warning, either upgrade or do the following:
>>> from cognite.client.config import global_config
>>> global_config.disable_pypi_version_check = True
  _threading_Thread_run(self)


## Check Login Status By Running Assets List

In [3]:
client.assets.list(limit=2)

,name,parent_id,metadata,id,created_time,last_updated_time,root_id,data_set_id
0,Vanuatu,5349541960763438,{},203912719934,2025-08-15 10:31:01.239,2025-08-15 10:31:01.239,1893265132227479,<NA>
1,French Guiana,5301790413104028,{},378116824183,2025-10-21 11:26:30.016,2025-10-21 11:26:30.016,6594401629304811,6522615934594328


## Start Your Own Code Here:

In [6]:
from cognite.client.data_classes import DataSet
client.data_sets.list()
prefix = "joe_ob"
ds = client.data_sets.create(DataSet(name=prefix+"_world_info",external_id=prefix+"_data_set"))
ds

,value
external_id,joe_ob_data_set
name,joe_ob_world_info
write_protected,False
id,7335814554953330
created_time,2025-10-15 18:07:34.204000
last_updated_time,2025-10-15 18:07:34.204000


In [24]:
from cognite.client.data_classes import DataSetUpdate
my_update = DataSetUpdate(id=7335814554953330).write_protected.set(False)
res = client.data_sets.update(my_update)

In [25]:
res

,value
external_id,joe_ob_data_set
name,joe_ob_world_info
metadata,{}
write_protected,False
id,7335814554953330
created_time,2025-10-15 18:07:34.204000
last_updated_time,2025-10-15 19:05:57.916000


In [93]:
from cognite.client.data_classes import AssetWrite
client.assets.create(AssetWrite(name="world", data_set_id=7335814554953330))

,value
name,world
data_set_id,7335814554953330
metadata,{}
id,3261792952088564
created_time,2025-10-15 22:00:29.194000
last_updated_time,2025-10-15 22:00:29.194000
root_id,3261792952088564


In [94]:
from cognite.client.data_classes import AssetUpdate
my_update = AssetUpdate(id=3261792952088564).name.set("global")
res = client.assets.update(my_update)

In [95]:
df = pd.read_csv("all.csv")
print(df.head())

             name alpha-2 alpha-3  country-code     iso_3166-2   region  \
0     Afghanistan      AF     AFG             4  ISO 3166-2:AF     Asia   
1   Åland Islands      AX     ALA           248  ISO 3166-2:AX   Europe   
2         Albania      AL     ALB             8  ISO 3166-2:AL   Europe   
3         Algeria      DZ     DZA            12  ISO 3166-2:DZ   Africa   
4  American Samoa      AS     ASM            16  ISO 3166-2:AS  Oceania   

        sub-region intermediate-region  region-code  sub-region-code  \
0    Southern Asia                 NaN        142.0             34.0   
1  Northern Europe                 NaN        150.0            154.0   
2  Southern Europe                 NaN        150.0             39.0   
3  Northern Africa                 NaN          2.0             15.0   
4        Polynesia                 NaN          9.0             61.0   

   intermediate-region-code  
0                       NaN  
1                       NaN  
2                       Na

In [97]:
clean_df = df.dropna(subset=['region'])
unique_values = clean_df['region'].unique()

for item in unique_values:
    print(item)

Asia
Europe
Africa
Oceania
Americas


In [100]:
from cognite.client.data_classes import AssetWrite
for item in unique_values:
    asset = AssetWrite(name=item, parent_id=3261792952088564, data_set_id=7335814554953330)
    client.assets.create(asset)

In [133]:
df_regions = client.assets.search(filter={"parent_ids": [3261792952088564]}).to_pandas()
df_regions

,name,parent_id,data_set_id,metadata,id,created_time,last_updated_time,root_id
0,Africa,3261792952088564,7335814554953330,{},6917770205399146,2025-10-15 22:06:20.255,2025-10-15 22:06:20.255,3261792952088564
1,Americas,3261792952088564,7335814554953330,{},5023569342550551,2025-10-15 22:59:16.775,2025-10-15 22:59:16.775,3261792952088564
2,Asia,3261792952088564,7335814554953330,{},8252367509583644,2025-10-15 22:06:19.897,2025-10-15 22:06:19.897,3261792952088564
3,Europe,3261792952088564,7335814554953330,{},2920044292887679,2025-10-15 22:06:20.075,2025-10-15 22:06:20.075,3261792952088564
4,Oceania,3261792952088564,7335814554953330,{},368024738085229,2025-10-15 22:06:20.438,2025-10-15 22:06:20.438,3261792952088564


In [103]:
lookup_value = 'Africa'
lookup_column = 'name'
target_column = 'id'
mask = df_regions[lookup_column] == lookup_value
target_id = df_regions.loc[mask, target_column].item()
print(target_id)

6917770205399146


In [134]:
assets=[]
for _, row in clean_df.iterrows():
    lookup_value = row['region']
    lookup_column = 'name'
    target_column = 'id'
    #print(lookup_value)
    mask = df_regions[lookup_column] == lookup_value
    #print(mask)
    target_id = df_regions.loc[mask, target_column].item()
    #print(target_id)
    assets.append(AssetWrite(name=row["name"], parent_id=target_id, data_set_id=7335814554953330))
    #print(assets)

client.assets.create(assets)

,name,parent_id,data_set_id,metadata,id,created_time,last_updated_time,root_id
0,Afghanistan,8252367509583644,7335814554953330,{},4269012503089690,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
1,Åland Islands,2920044292887679,7335814554953330,{},8985822931507761,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
2,Albania,2920044292887679,7335814554953330,{},7318521408197035,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
3,Algeria,6917770205399146,7335814554953330,{},270669894584083,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
4,American Samoa,368024738085229,7335814554953330,{},5994021344131589,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
...,...,...,...,...,...,...,...,...
242,Wallis and Futuna,368024738085229,7335814554953330,{},3029410809682247,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
243,Western Sahara,6917770205399146,7335814554953330,{},8412714252518581,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
244,Yemen,8252367509583644,7335814554953330,{},1598941891247718,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
245,Zambia,6917770205399146,7335814554953330,{},2324636893050808,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564


In [139]:
client.assets.search(filter={"parent_ids": [368024738085229]})

,name,parent_id,data_set_id,metadata,id,created_time,last_updated_time,root_id
0,American Samoa,368024738085229,7335814554953330,{},5994021344131589,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
1,Australia,368024738085229,7335814554953330,{},5069133151729308,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
2,Christmas Island,368024738085229,7335814554953330,{},3643201862345265,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
3,Cocos (Keeling) Islands,368024738085229,7335814554953330,{},5849259310496193,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
4,Cook Islands,368024738085229,7335814554953330,{},3996674196001134,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
5,Fiji,368024738085229,7335814554953330,{},3723718148949408,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
6,French Polynesia,368024738085229,7335814554953330,{},5682472394868798,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
7,Guam,368024738085229,7335814554953330,{},1608428618270218,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
8,Heard Island and McDonald Islands,368024738085229,7335814554953330,{},3232770246134328,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564
9,Kiribati,368024738085229,7335814554953330,{},8134125840451243,2025-10-15 23:00:07.213,2025-10-15 23:00:07.213,3261792952088564


In [186]:
from cognite.client.data_classes import TimeSeriesWrite

df_country_assets = client.assets.list(parent_ids=[6917770205399146,5023569342550551,8252367509583644,2920044292887679,368024738085229],limit=-1).to_pandas()

ts=[]

for _, row in df_country_assets.iterrows():
    ts.append(TimeSeriesWrite(name=row['name']+'_population', asset_id=row['id'], data_set_id=7335814554953330))

client.time_series.create(ts)

,name,is_string,metadata,asset_id,is_step,security_categories,data_set_id,id,created_time,last_updated_time
0,Turks and Caicos Islands_population,False,{},74743603082318,False,[],7335814554953330,5228175672586532,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
1,Congo_population,False,{},94460664537201,False,[],7335814554953330,2224771565656204,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
2,United Arab Emirates_population,False,{},109241323385490,False,[],7335814554953330,8004092083370245,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
3,Bahamas_population,False,{},127471693393507,False,[],7335814554953330,5117027712420199,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
4,Uganda_population,False,{},162071988635294,False,[],7335814554953330,4653295912986475,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
...,...,...,...,...,...,...,...,...,...,...
243,Virgin Islands (British)_population,False,{},8880469707318035,False,[],7335814554953330,1170014289937176,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
244,Marshall Islands_population,False,{},8910152218570852,False,[],7335814554953330,4350232551671213,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
245,Palau_population,False,{},8927647029652717,False,[],7335814554953330,7356337540514295,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961
246,Åland Islands_population,False,{},8985822931507761,False,[],7335814554953330,6397486079930625,2025-10-17 16:43:34.961,2025-10-17 16:43:34.961


In [212]:
df_populations = pd.read_csv('../data/populations_postprocessed.csv')
df_populations.columns = df_populations.columns + '_population'
# Find the mapping of the time series to its id
ts_to_id = client.time_series.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()[['name','id']].set_index('name')['id'].to_dict()

# Rename data frame so that columns correspond to the time seris id
df_populations = df_populations.rename(columns=ts_to_id)

df_populations['Unnamed: 0_population'] = pd.to_datetime(df_populations['Unnamed: 0_population'])
df_populations = df_populations.set_index('Unnamed: 0_population')
df_populations.head()


,3618396330788897,8965145047222557,2766775005920685,8911327919253454,4991892156625279,8004092083370245,3824630468921814,2847780117682463,6414378743723079,6062662741880158,...,4653295912986475,8519105606722058,3166157478790540,1402388318086151,6695206606317487,6871178152234874,3165374837266002,1250425371349774,8350732306419532,7204243398796248
Unnamed: 0_population,,,,,,,,,,,,,,,,,,,,,
1990-01-01,62152.0,12412311.0,11848385.0,3286542.0,54508.0,1828437.0,32618648.0,3538164.0,47351.0,62533.0,...,17354395.0,51891400.0,3109598.0,20510000.0,103963.0,146575.0,162797.0,36800507.0,8036849.0,10432409.0
1991-01-01,64623.0,13299016.0,12248901.0,3266790.0,56666.0,1937159.0,33079002.0,3505249.0,48682.0,63363.0,...,17953534.0,52000500.0,3131657.0,20952000.0,104807.0,150718.0,164000.0,37718952.0,8246662.0,10681008.0
1992-01-01,68240.0,14485543.0,12657361.0,3247039.0,58882.0,2052892.0,33529320.0,3442820.0,49900.0,64459.0,...,18561668.0,52150400.0,3154459.0,21449000.0,105712.0,155176.0,165490.0,38672611.0,8451346.0,10900511.0
1993-01-01,72495.0,15816601.0,13075044.0,3227287.0,60974.0,2173135.0,33970103.0,3363111.0,51025.0,65777.0,...,19175986.0,52179200.0,3177734.0,21942000.0,106578.0,159743.0,167117.0,39633754.0,8656484.0,11092775.0
1994-01-01,76705.0,17075728.0,13503753.0,3207536.0,62676.0,2294377.0,34402669.0,3283664.0,52099.0,67201.0,...,19793541.0,51921400.0,3201149.0,22377000.0,107318.0,164128.0,168689.0,40564061.0,8869745.0,11261752.0


In [214]:
#print(df_populations.columns.tolist())
client.time_series.data.insert_dataframe(df_populations, external_id_headers=False)

In [ ]:
from cognite.client.data_classes import filters

search_list = ["Latvia_population", "Guatemala_population", "Benin_population"]
adv_filter = filters.And(
    filters.In("name", search_list), 
    filters.Equals("data_set_id", 7335814554953330)
)
df_selected = client.time_series.filter(filter=adv_filter).to_pandas()
ds_ids = df_selected['id'].to_list()
print(ds_ids)
for item in ds_ids:
    datapoints = client.time_series.data.retrieve(id=item)
    print(datapoints)

In [236]:
adv_filter = filters.And(
    filters.Equals("parent_id", 2920044292887679),
    filters.Equals("data_set_id", 7335814554953330)
)
df_assets_in_eu = client.assets.filter(adv_filter,limit=-1).to_pandas()
assetIds_in_eu = df_assets_in_eu["id"].to_list()
print(assetIds_in_eu)
# List all time series of those assets
ts = client.time_series.list(data_set_ids=[7335814554953330],asset_ids=assetIds_in_eu, limit=-1)

# Retrieve the latest data for all of these time series
data = client.time_series.data.retrieve_latest([item.id for item in ts]).to_pandas()
data.head()
print(data.T.sum())

[418458002943973, 419620989214339, 505898643682321, 570311706761061, 588661809286109, 699868508622706, 948730200359730, 1515081802499048, 1611720636982057, 1978740915752198, 2548045025854882, 2849497442064217, 2969708084869374, 2990831008750452, 3064055882579111, 3156334080549739, 4024890163683193, 4085641824781832, 4191375203846295, 4268859580594480, 4463536705424744, 4563476888498435, 4619550905193132, 4666643101897704, 4678872124298553, 5576839006277364, 5638053639391281, 5644618907101650, 5722576523631642, 6001582726216769, 6297770812476193, 6634264154966360, 6754486054195044, 6796563220910321, 7132133650445593, 7159136754192416, 7174695683462435, 7236348311896187, 7302812702672211, 7318521408197035, 7626235710438187, 7822377665915119, 7839381849622381, 8157773008181595, 8297170099627234, 8330481482256733, 8409647551283784, 8537013786205839, 8686800262672064, 8859915377470182, 8985822931507761]
2021-01-01    656621484.0
dtype: float64


In [257]:
from pathlib import Path

folder_path = Path('../data/files')

# Use a list comprehension to iterate over all entries and filter for files
country_names_with_uploads = [
    p.name.removesuffix('.pdf') for p in folder_path.iterdir() if p.is_file()
]

print(f"COUNT: {len(country_names_with_uploads)}")

COUNT: 25


In [261]:
df_assets = client.assets.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()

mask = df_assets['name'].isin(country_names_with_uploads)

df_matching_asset_ids = df_assets.loc[mask, ['id', 'name']]
df_matching_asset_ids

,id,name
36,1382017945922732,Honduras
43,1598941891247718,Yemen
45,1606994126893679,Tokelau
51,1978740915752198,United Kingdom of Great Britain and Northern I...
52,2019924438529673,Vanuatu
54,2058557760485138,Rwanda
58,2137227535917371,Comoros
60,2304923158120065,Liberia
61,2324636893050808,Zambia
75,2849497442064217,San Marino


In [278]:
upload_result = df_matching_asset_ids.apply(
    lambda row: client.files.upload(
        # Pass the file path from the 'file_path' column
        path=f"../data/files/{row['name']}.pdf", 
            
        # Pass the mandatory metadata parameters
        name=f"{row['name']}_data_sheet",
        data_set_id=7335814554953330,
        asset_ids=[row['id']],
            
        # Set other optional parameters as needed
        mime_type='application/pdf', 
            
        # If you want to replace existing files, set this to True
        overwrite=False 
    ),
    # Apply the function across rows (axis=1)
    axis=1
)
    
print(f"Successfully started {len(upload_result)} file uploads.")

Successfully started 25 file uploads.


In [ ]:
df_files = client.files.list(asset_ids=[2019924438529673]).to_pandas()
df_files

In [331]:
df_events = pd.read_csv('../data/events.csv', keep_default_na=False)
print(df_events)

           Dis No Disaster Group Disaster Subgroup        Disaster Type  \
0   2011-0159-ESP        Natural       Geophysical           Earthquake   
1   2010-0170-ISL        Natural       Geophysical    Volcanic activity   
2   2010-0574-SRB        Natural       Geophysical           Earthquake   
3   2012-0152-BGR        Natural       Geophysical           Earthquake   
4   2013-0121-HUN        Natural       Geophysical           Earthquake   
5   2012-0142-ITA        Natural       Geophysical           Earthquake   
6   2012-0162-ITA        Natural       Geophysical           Earthquake   
7   2014-0049-GRC        Natural       Geophysical           Earthquake   
8   2014-0174-GRC        Natural       Geophysical           Earthquake   
9   2017-0182-GRC        Natural       Geophysical           Earthquake   
10  2017-0280-GRC        Natural       Geophysical           Earthquake   
11  2017-0350-CHE        Natural       Geophysical  Mass movement (dry)   
12  2016-0313-ITA        

In [332]:
def find_asset_ids_by_country_names(country_name_list):
    df_assets = client.assets.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()
    mask = df_assets['name'].isin(country_name_list)
    return df_assets.loc[mask, ['id', 'name']]

country_names = df_events['Country'].unique().tolist()
df_asset_lookup = find_asset_ids_by_country_names(country_names)
df_asset_lookup.rename(columns={'name': 'Country', 'id': 'Asset ID'}, inplace=True)
df_events = df_events.merge(
    df_asset_lookup,
    on='Country',
    how='left'
)
df_events

,Dis No,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,Event Name,Country,ISO,Region,Continent,Location,Origin,Start Year,Start Month,Start Day,End Year,End Month,End Day,Asset ID
0,2011-0159-ESP,Natural,Geophysical,Earthquake,Ground movement,,Spain,ESP,Southern Europe,Europe,"Lorca municipality (Murcia district, Region de...",,2011,5,11,2011,5,11,4563476888498435
1,2010-0170-ISL,Natural,Geophysical,Volcanic activity,Ash fall,Eyjafjallajokull,Iceland,ISL,Northern Europe,Europe,"Fljotshlid, Eyjafjöll, and Landeyjar villages...",,2010,3,1,2010,4,28,6297770812476193
2,2010-0574-SRB,Natural,Geophysical,Earthquake,Ground movement,,Serbia,SRB,Southern Europe,Europe,Kraljevo town (Raski Province),,2010,11,3,2010,11,3,6796563220910321
3,2012-0152-BGR,Natural,Geophysical,Earthquake,Ground movement,,Bulgaria,BGR,Eastern Europe,Europe,"Pernik town (Pernik Province), Sofia town (Sof...",,2012,5,22,2012,5,22,8859915377470182
4,2013-0121-HUN,Natural,Geophysical,Earthquake,Ground movement,,Hungary,HUN,Eastern Europe,Europe,Heves province,,2013,4,14,2013,4,23,3156334080549739
5,2012-0142-ITA,Natural,Geophysical,Earthquake,Ground movement,,Italy,ITA,Southern Europe,Europe,"Finale Emilia, Mirandola towns (Modena Provinc...",,2012,5,20,2012,5,20,699868508622706
6,2012-0162-ITA,Natural,Geophysical,Earthquake,Ground movement,,Italy,ITA,Southern Europe,Europe,Modena Province (Emilia-romagna Region),,2012,5,29,2012,5,29,699868508622706
7,2014-0049-GRC,Natural,Geophysical,Earthquake,Ground movement,,Greece,GRC,Southern Europe,Europe,"Lixouri, Kounopetra, Fiskardo villages (Kefalo...",,2014,1,26,2014,2,3,2548045025854882
8,2014-0174-GRC,Natural,Geophysical,Earthquake,Ground movement,,Greece,GRC,Southern Europe,Europe,"Limnos Island (Lesvou district, Voreio Aigaio ...",,2014,5,24,2014,5,24,2548045025854882
9,2017-0182-GRC,Natural,Geophysical,Earthquake,Ground movement,,Greece,GRC,Southern Europe,Europe,"Vrisa, Plomarion,Plagias, Chios, Kampas, Skala...",,2017,6,12,2017,6,12,2548045025854882


In [345]:
from cognite.client.data_classes import EventWrite
from dateutil.parser import parse
from dateutil.parser import ParserError
from datetime import datetime, timezone

def is_valid_date(date_string):
    # Checks if a string can be parsed into a valid date object
    try:
        # Attempt to parse the string into a datetime object
        parse(date_string)
        return True
    except (ValueError, TypeError, ParserError):
        # Catches common errors if the string is not a valid date
        return False

def convert_to_epoch_format(date_str):
    date_format = "%Y-%m-%d"
    dt_object = datetime.strptime(date_str, date_format)
    epoch_seconds = dt_object.replace(tzinfo=timezone.utc).timestamp()
    epoch_milliseconds = int(epoch_seconds * 1000)
    return epoch_milliseconds

def create_event_obj(start_date, end_date, dis_type, dis_stype, loc, asset_id):
    if is_valid_date(start_date) and is_valid_date(end_date):
        start_date_int = convert_to_epoch_format(start_date)
        end_date_int = convert_to_epoch_format(end_date)
    else:
        print("failed in conversion")
        return False
    dict_loc = {"Location": loc}
    return EventWrite(start_time=start_date_int, end_time=end_date_int, type=dis_type, subtype=dis_stype, metadata=dict_loc, data_set_id=7335814554953330, asset_ids=[asset_id])

def create_event_writes(df) -> list[EventWrite]:
    list_of_dicts = df.to_dict('records')
    event_list = [
        create_event_obj(
            f"{row['Start Year']}-{row['Start Month']}-{row['Start Day']}",
            f"{row['End Year']}-{row['End Month']}-{row['End Day']}",
            row['Disaster Type'],
            row['Disaster Subtype'],
            row['Location'],
            row['Asset ID']
        )
        for row in list_of_dicts
    ]    
    return event_list

list_of_event_writes = create_event_writes(df_events)
res = client.events.create(list_of_event_writes)

In [351]:
adv_filter = filters.And(
    filters.Equals("data_set_id", 7335814554953330),
    filters.Equals("type", "Volcanic activity")
)
volcanic_events_list = client.events.filter(adv_filter)
volcanic_events_list

,data_set_id,start_time,end_time,type,subtype,metadata,asset_ids,id,created_time,last_updated_time
0,7335814554953330,2010-03-01,2010-04-28,Volcanic activity,Ash fall,"{'Location': ' Fljotshlid, Eyjafjöll, and Land...",[6297770812476193],5748496791308401,2025-10-20 22:07:08.685,2025-10-20 22:07:08.685
1,7335814554953330,2018-12-24,2018-12-26,Volcanic activity,None,"{'Location': 'Acicatena, Acireale, Aci Sant’An...",[699868508622706],1898901976672514,2025-10-20 22:00:08.365,2025-10-20 22:00:08.365
2,7335814554953330,2018-12-24,2018-12-26,Volcanic activity,None,"{'Location': 'Acicatena, Acireale, Aci Sant’An...",[699868508622706],4090514054282635,2025-10-20 21:58:21.123,2025-10-20 21:58:21.123
3,7335814554953330,2018-12-24,2018-12-26,Volcanic activity,None,"{'Location': 'Acicatena, Acireale, Aci Sant’An...",[699868508622706],8856878275470525,2025-10-20 22:07:08.685,2025-10-20 22:07:08.685


In [ ]:
res = client.time_series.search(name='Netherlands')
print(res)
from cognite.client.data_classes import TimeSeriesUpdate
my_update = TimeSeriesUpdate(id=2527585060365203).name.set("Netherlands_population")
res = client.time_series.update(my_update)
print(res)
#tss = TimeSeriesWrite(name='Netherlands_population', asset_id=row['id'], data_set_id=7335814554953330)

#client.time_series.create(tss)

In [7]:
client.labels.list(name="Hot")

,external_id,name,description,created_time
0,Henrik_labels_hot,Hot,This is hot. The original label seems to be gone.,2025-08-18 09:02:01.306


In [7]:
client.labels.list(name="Hot")

,external_id,name,description,created_time
0,Henrik_labels_hot,Hot,This is hot. The original label seems to be gone.,2025-08-18 09:02:01.306


In [8]:
#df = client.assets.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()
#res = client.assets.delete(id=df["id"].tolist())
#df = client.time_series.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()
#res = client.time_series.delete(id=df["id"].tolist())
#df = client.files.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()
#res = client.files.delete(id=df["id"].tolist())
df = client.events.list(data_set_ids=[7335814554953330], limit=-1).to_pandas()
res = client.events.delete(id=df["id"].tolist())

In [9]:
res